# Titanic Survival Prediction
Example of using MLflow to log experiments and register models.


In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [0]:
# Load data set

data_url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
df = pd.read_csv(data_url)

# Define features (X) and target variable (y), dropping rows with missing target values

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
target = 'Survived'

df = df.dropna(subset=[target])

X = df[features]
y = df['Survived']

# Split into 80% training and 20% testing data

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [0]:
# Set up preprocessing and model pipeline

numeric_features = ['Age', 'SibSp', 'Parch', 'Fare']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_features = ['Sex', 'Embarked']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [0]:
# Enable MLflow auto-logging
mlflow.sklearn.autolog(log_models=True, exclusive=False)

run_name = "titanic_run"

with mlflow.start_run(run_name=run_name) as run:
    # Capture the active run information
    active_run_id = run.info.run_id
    print(f"Active Run ID: {active_run_id}")
    
    # Configure custom parameters inside our pipeline step
    n_estimators = 100
    max_depth = 5
    pipeline.set_params(classifier__n_estimators=n_estimators, classifier__max_depth=max_depth)
    
    # Train the model (Autologging captures everything here)
    pipeline.fit(X_train, y_train)
    
    # Generate predictions for custom evaluations
    y_pred = pipeline.predict(X_test)
    accuracy = pipeline.score(X_test, y_test)
    print(f"Test Accuracy: {accuracy:.4f}")
    
    # Explicitly log metadata points
    mlflow.log_param("dataset_source_url", data_url)
    mlflow.log_metric("test_accuracy", accuracy)
    
    # Generate a visual performance chart artifact
    fig, ax = plt.subplots(figsize=(6, 6))
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(ax=ax, cmap="Blues")
    plt.title("Titanic Test Data Confusion Matrix")
    
    plot_path = "confusion_matrix.png"
    plt.savefig(plot_path)
    plt.close(fig)
    
    # Log the structural plot file directly into the MLflow run artifact store
    mlflow.log_artifact(plot_path)
    print("Custom evaluation plot successfully logged to MLflow artifacts.")


In [0]:
model_name = "titanic_survival_prediction_model"
model_uri = f"runs:/{active_run_id}/model"

print(f"Registering model from URI: {model_uri}")

# Programmatically register the model object
registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)

print(f"Model successfully registered under name: '{registered_model.name}'")
print(f"Current version of this deployment track: Version {registered_model.version}")